In [11]:
import pandas as pd
import numpy as np
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

from sklearn.metrics import classification_report, accuracy_score


In [12]:
X_train = joblib.load("../models/X_train.pkl")
X_test = joblib.load("../models/X_test.pkl")
y_train = joblib.load("../models/y_train.pkl")
y_test = joblib.load("../models/y_test.pkl")


In [13]:
df = pd.read_csv("../data/processed/cleaned_comments.csv")

X = df["cleaned_text"]
y = df["is_toxic"]

vectorizer = joblib.load("../models/vectorizer.pkl")
X_tfidf = vectorizer.transform(X)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)


In [14]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

print("Logistic Regression Accuracy:",
      accuracy_score(y_test, y_pred_lr))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_lr))


Logistic Regression Accuracy: 0.9547606357964699

Classification Report:

              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28631
           1       0.93      0.61      0.73      3266

    accuracy                           0.95     31897
   macro avg       0.94      0.80      0.85     31897
weighted avg       0.95      0.95      0.95     31897



In [15]:
nb_model = MultinomialNB()

nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("Naive Bayes Accuracy:",
      accuracy_score(y_test, y_pred_nb))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_nb))


Naive Bayes Accuracy: 0.9476439790575916

Classification Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.97     28631
           1       0.93      0.53      0.67      3266

    accuracy                           0.95     31897
   macro avg       0.94      0.76      0.82     31897
weighted avg       0.95      0.95      0.94     31897



In [16]:
svm_model = LinearSVC()

svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

print("SVM Accuracy:",
      accuracy_score(y_test, y_pred_svm))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_svm))


SVM Accuracy: 0.9579270777816096

Classification Report:

              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28631
           1       0.89      0.68      0.77      3266

    accuracy                           0.96     31897
   macro avg       0.93      0.83      0.87     31897
weighted avg       0.96      0.96      0.96     31897



In [17]:
results = pd.DataFrame({
    "Model": ["Logistic Regression", "Naive Bayes", "SVM"],
    "Accuracy": [
        accuracy_score(y_test, y_pred_lr),
        accuracy_score(y_test, y_pred_nb),
        accuracy_score(y_test, y_pred_svm)
    ]
})

results


,Model,Accuracy
0,Logistic Regression,0.954761
1,Naive Bayes,0.947644
2,SVM,0.957927


In [18]:
joblib.dump(lr_model, "../models/toxic_model.pkl")

print("✅ Final model saved successfully!")


✅ Final model saved successfully!


In [19]:
def predict_comment(text):
    cleaned = text.lower()
    vec = vectorizer.transform([cleaned])
    pred = lr_model.predict(vec)[0]
    return "Toxic" if pred == 1 else "Non-Toxic"

predict_comment("You are absolutely stupid")


'Toxic'

In [20]:
from sklearn.multiclass import OneVsRestClassifier

multi_model = OneVsRestClassifier(LogisticRegression(max_iter=1000))

multi_model.fit(X_train, y_train)

y_pred = multi_model.predict(X_test)

print(classification_report(y_test, y_pred))



joblib.dump(multi_model, "../models/toxic_model.pkl")


              precision    recall  f1-score   support

           0       0.96      0.99      0.98     28631
           1       0.93      0.61      0.73      3266

    accuracy                           0.95     31897
   macro avg       0.94      0.80      0.85     31897
weighted avg       0.95      0.95      0.95     31897



['../models/toxic_model.pkl']